# 7. 알고리즘 - 깊이우선탐색과 너비우선탐색

재귀에서 배운 "자기 자신을 다시 부르기"를 그래프 위에서 쓰면 **깊이우선탐색(DFS)** 이 된다.
같은 일을 큐로 하면 **너비우선탐색(BFS)** 이 된다.

## 두 탐색의 차이
- **DFS(Depth First Search):** 갈 수 있는 데까지 깊이 들어갔다가 막히면 되돌아온다. **스택**(재귀의 호출 스택 포함)을 쓴다.
- **BFS(Breadth First Search):** 가까운 곳부터 한 겹씩 넓혀 나간다. **큐**를 쓴다.

## 공통 뼈대
어느 쪽이든 하는 일은 똑같다.
1. 시작점을 방문 표시한다.
2. 아직 방문하지 않은 이웃으로 간다.
3. 더 갈 곳이 없으면 끝낸다.

**방문 표시(visited)를 빼먹으면 같은 곳을 무한히 돈다.** 이게 가장 흔한 실수다.

## 푸는 방법
- **[직접 풀기]** 칸: 빈칸/시그니처만 있다. 직접 채운다.
- **[정답]** 칸: 완성 코드. 풀어본 뒤 비교한다.
- **출력 예측** 문제는 실행 전에 결과를 먼저 종이에 적는다.

## 목록
1. 그래프를 딕셔너리로 표현하기
2. 깊이우선탐색 - 재귀
3. 깊이우선탐색 - 스택 (반복)
4. 너비우선탐색 - 큐
5. BFS로 최단 거리 구하기
6. 격자 탐색 (섬 세기)


---
## 1. 그래프를 딕셔너리로 표현하기

**그래프**는 점(정점, vertex)과 선(간선, edge)의 모임이다. 이 수업에서 쓸 그래프:

```
    0 --- 1 --- 3
    |     |
    2     4
     \   /
       5
```

**정점을 키(key), 그 정점의 이웃 목록을 값(value)** 으로 하는 딕셔너리로 적는다.

```python
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}
```

`graph[2]`는 `[0, 5]` - 2번 정점의 이웃은 0번과 5번이라는 뜻이다.
연결은 양쪽에 다 적는다(무방향 그래프). 0의 목록에 1이 있으면 1의 목록에도 0이 있다.

## 왜 리스트가 아니라 딕셔너리인가
- 정점 번호가 `0, 1, 2, ...` 처럼 **연속일 필요가 없다.** `10`, `77` 만 있어도 된다.
- 번호 대신 `"서울"`, `"부산"` 같은 **이름을 키로 써도 그대로 동작한다.**
- `for v in graph:` 로 돌면 **키(정점)** 가 하나씩 나온다.
- `v in graph` 는 "그런 정점이 있는가"를 확인한다.


In [ ]:
# 연습용 코드
# 그래프 직접 만들기
graph = {
    0: [1, 2, 4],
    1: [],
    2: [3, 4],
    3: [],
    4: [2, 3]
}

**(1-1) 출력 예측** - 아래 코드의 출력을 먼저 예측한 뒤 실행한다.

In [ ]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

print(graph[1])        # 예측: ?
print(len(graph[3]))   # 예측: ?
print(graph[5][0])     # 예측: ?
print(len(graph))      # 예측: ?

**정답:** `[0, 3, 4]`, `1`, `2`, `6`
```
graph[1]      -> 1번의 이웃 목록 그대로   -> [0, 3, 4]
len(graph[3]) -> 3번의 이웃은 [1] 하나뿐  -> 1
graph[5][0]   -> 5번의 이웃 [2,4]의 0번째 -> 2
len(graph)    -> 딕셔너리의 키 개수 = 정점 수 -> 6
```
`len(graph)`는 **간선 수가 아니라 정점 수**다. 헷갈리지 말 것.

**(1-2) 빈칸 - 차수 세기** 각 정점에 연결된 간선 수(차수)를 출력하는 코드다. 두 곳을 채워라.

In [7]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

# 다른 방법으로 나가는 차수 + 들어오는 차수(위상) 구하기
degree = [0]*len(graph)     # 차수 리스트 초기화
for v in graph:
    for i in graph[v]:
        degree[v] += 1
        degree[i] += 1
print(degree)

[4, 6, 4, 2, 4, 4]
[4, 6, 4, 2, 4, 4]


In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

for v in graph:                              # for v in graph.keys() 와 같다
    print(v, "번 정점의 차수:", len(graph[v]))

---
## 2. 깊이우선탐색 - 재귀

재귀 함수의 뼈대를 그대로 쓴다. 방문한 정점은 **리스트 `visited`에 순서대로 쌓는다.**

```python
visited = []

def dfs(v):
    visited.append(v)          # 1. 나를 방문 표시하고
    for nxt in graph[v]:       # 2. 이웃들에 대해
        if nxt not in visited: #    아직 안 갔으면
            dfs(nxt)           #    거기서 다시 시작
```

`visited`에 **방문 순서가 그대로 기록**되므로 끝나고 출력하면 탐색 순서를 볼 수 있다.

**기저 조건이 눈에 안 보이지만 있다.** 이웃이 전부 방문 완료면 `for`문이 아무것도 하지 않고 함수가 끝난다.
그게 멈춤 지점이다.

**(2-1) 출력 예측** 0번에서 DFS를 시작하면 `visited`는 어떤 순서가 될까? 그림을 보고 손으로 따라가라.

In [ ]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []
def dfs(v): # 깊이 우선 탐색 depth first search
    visited.append(v)
    for nxt in graph[v]:
        if nxt not in visited:
            dfs(nxt)

dfs(0)
print(visited)   # 예측: ?

**정답:** `[0, 1, 3, 4, 5, 2]`
```
dfs(0) 0 방문 -> 이웃 1,2 중 1로
  dfs(1) 1 방문 -> 이웃 0(방문함), 3, 4 중 3으로
    dfs(3) 3 방문 -> 이웃 1(방문함) 뿐 -> 끝, 되돌아감
  dfs(1)로 복귀 -> 다음 이웃 4로
    dfs(4) 4 방문 -> 이웃 1(방문함), 5 중 5로
      dfs(5) 5 방문 -> 이웃 2, 4(방문함) 중 2로
        dfs(2) 2 방문 -> 이웃 0,5 둘 다 방문함 -> 끝
dfs(0)으로 복귀 -> 남은 이웃 2는 이미 방문함 -> 끝
```
**이웃 목록에 적은 순서대로** 들어간다. 순서를 바꾸면 방문 순서도 바뀐다.

**(2-2) 빈칸 - 방문 표시** 두 곳을 채워라. 이게 빠지면 무한 재귀에 빠진다.

In [ ]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    ____                    # 방문 표시
    for nxt in graph[v]:
        if ____:            # 아직 안 간 이웃만
            dfs(nxt)

# dfs(0)
# print(visited)   # [0, 1, 3, 4, 5, 2]

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    visited.append(v)          # 방문 순서대로 쌓인다
    for nxt in graph[v]:
        if nxt not in visited: # 리스트 안에 없으면 아직 안 간 것
            dfs(nxt)

dfs(0)
print(visited)

**(2-3) 디버깅 - 무한 재귀** 아래 코드는 멈추지 않는다(RecursionError). 이유를 말하고 고쳐라.
**실행하면 에러가 나므로** 먼저 눈으로 찾아보라.

In [ ]:
# [직접 풀기] - 어디가 문제일까? (실행하면 RecursionError)
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    print(v, end=" ")
    for nxt in graph[v]:
        if nxt not in visited:
            dfs(nxt)

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    visited.append(v)   # 빠져 있던 줄. 방문 표시를 안 하면 0->1->0->1... 을 영원히 반복한다
    print(v, end=" ")
    for nxt in graph[v]:
        if nxt not in visited:
            dfs(nxt)

dfs(0)

**(2-4) 전체 작성 - 도달 가능한 정점 세기** 0번에서 출발해 도달할 수 있는 정점의 **개수**를 세어
반환하는 함수를 작성하라. (위 그래프는 전부 이어져 있으므로 답은 6)

In [ ]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    pass

# print(dfs(0))   # 6

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []

def dfs(v):
    visited.append(v)
    count = 1                          # 나 자신 1개
    for nxt in graph[v]:
        if nxt not in visited:
            count = count + dfs(nxt)   # 이웃 쪽에서 센 개수를 더한다
    return count

print(dfs(0))        # 6
print(len(visited))  # 6 - visited의 길이를 재도 같다

---
## 3. 깊이우선탐색 - 스택 (반복)

재귀는 **호출 스택**을 자동으로 써 준다. 그 스택을 우리가 직접 리스트로 만들면 재귀 없이 DFS를 할 수 있다.

```python
visited = []
stack = [start]
while stack:
    v = stack.pop()            # 맨 뒤에서 꺼낸다 (LIFO)
    if v in visited:
        continue               # 이미 갔던 곳이면 건너뛴다
    visited.append(v)
    for nxt in graph[v]:
        if nxt not in visited:
            stack.append(nxt)
```

**주의:** 방문 순서가 재귀 버전과 다를 수 있다. 스택은 나중에 넣은 것을 먼저 꺼내므로
이웃 목록의 **뒤쪽부터** 들어간다. 하지만 "깊이 먼저"라는 성질은 똑같다.

**(3-1) 출력 예측** 스택 버전의 방문 순서를 예측하라. 재귀 버전(`[0, 1, 3, 4, 5, 2]`)과 비교해보라.

In [8]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []
stack = [0]
while stack:
    v = stack.pop()
    if v in visited:
        continue
    visited.append(v)
    for nxt in graph[v]:
        if nxt not in visited:
            stack.append(nxt)
graph = { 0: [1,2], 1: [2], 2: [0] }
'''
visited     stack       v       graph[v]        nxt
[]          [0]
            []          0       [1,2]
[0]         []          0       [1,2]           1
[0]         [1]                                 2
[0]         [1,2]                               2

[0]         [1]         2       [0]
[0,2]       [1]         2       [0]             0

[0,2]       []          1       [2]
[0,2,1]     []          1       [2]             2


'''

[0, 2, 5, 4, 1, 3]


**정답:** `[0, 2, 5, 4, 1, 3]`
```
stack=[0]      pop 0 -> 방문 0, 이웃 1,2 push  -> stack=[1,2]
stack=[1,2]    pop 2 -> 방문 2, 이웃 5 push    -> stack=[1,5]
stack=[1,5]    pop 5 -> 방문 5, 이웃 4 push    -> stack=[1,4]
stack=[1,4]    pop 4 -> 방문 4, 이웃 1 push    -> stack=[1,1]
stack=[1,1]    pop 1 -> 방문 1, 이웃 3 push    -> stack=[1,3]
stack=[1,3]    pop 3 -> 방문 3                 -> stack=[1]
stack=[1]      pop 1 -> 이미 방문 -> continue  -> stack=[]
```
재귀 버전과 순서가 다르다. **둘 다 정상적인 DFS다.**
스택에 같은 번호가 중복으로 들어갈 수 있어서 꺼낸 뒤 `if v in visited: continue` 로 걸러준다.

**(3-2) 빈칸 - 스택 연산** 두 곳을 채워라. 스택은 어느 쪽에서 넣고 꺼내는가?

In [ ]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []
stack = [0]
while stack:
    v = stack.____()          # 스택에서 꺼내기
    if v in visited:
        continue
    visited.append(v)
    for nxt in graph[v]:
        if nxt not in visited:
            stack.____(nxt)   # 스택에 넣기

# print(visited)   # [0, 2, 5, 4, 1, 3]

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = []
stack = [0]
while stack:
    v = stack.pop()           # 맨 뒤에서 꺼낸다 (LIFO)
    if v in visited:
        continue
    visited.append(v)
    for nxt in graph[v]:
        if nxt not in visited:
            stack.append(nxt) # 맨 뒤에 넣는다

print(visited)

---
## 4. 너비우선탐색 - 큐

스택을 **큐**로 바꾸기만 하면 BFS가 된다. 코드 구조는 DFS 스택 버전과 거의 같다.

```python
visited = [start]             # 넣을 때 바로 방문 표시하는 것이 요령
queue = [start]
while queue:
    v = queue.pop(0)          # 맨 앞에서 꺼낸다 (FIFO)
    for nxt in graph[v]:
        if nxt not in visited:
            visited.append(nxt)
            queue.append(nxt)
```

**차이는 `pop()` 이냐 `pop(0)` 이냐, 이 하나뿐이다.**
BFS는 **큐에 넣는 순간 방문 표시**를 한다. 그래야 같은 정점이 큐에 두 번 들어가지 않는다.

**(4-1) 출력 예측** 0번에서 BFS를 시작하면 `visited`는 어떤 순서가 될까?

In [ ]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = [0]
queue = [0]
while queue:
    v = queue.pop(0)
    for nxt in graph[v]:
        if nxt not in visited:
            visited.append(nxt)
            queue.append(nxt)

print(visited)   # 예측: ?

In [ ]:
graph = {
    0: [1],
    1: [2],
    2: [3],
    3: [0]
}

**정답:** `[0, 1, 2, 3, 4, 5]`
```
queue=[0]      pop 0 -> 이웃 1,2 넣음   -> queue=[1,2]
queue=[1,2]    pop 1 -> 이웃 3,4 넣음   -> queue=[2,3,4]
queue=[2,3,4]  pop 2 -> 이웃 5 넣음     -> queue=[3,4,5]
queue=[3,4,5]  pop 3 -> 새 이웃 없음    -> queue=[4,5]
queue=[4,5]    pop 4 -> 새 이웃 없음    -> queue=[5]
queue=[5]      pop 5 -> 새 이웃 없음    -> queue=[]
```
0에서 거리 0인 것(0), 거리 1인 것(1,2), 거리 2인 것(3,4,5) 순서로 나온다.
**이게 BFS의 핵심 성질이다.**

**(4-2) 빈칸 - 큐 연산** DFS 스택 버전에서 딱 한 군데만 다르다. 두 빈칸을 채워라.

In [ ]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = [0]
queue = [0]
while queue:
    v = queue.____        # 큐에서 꺼내기 (맨 앞)
    for nxt in graph[v]:
        if nxt not in visited:
            ____          # 넣을 때 바로 방문 표시
            queue.append(nxt)

# print(visited)   # [0, 1, 2, 3, 4, 5]

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = [0]
queue = [0]
while queue:
    v = queue.pop(0)           # FIFO: 맨 앞에서 꺼낸다
    for nxt in graph[v]:
        if nxt not in visited:
            visited.append(nxt)
            queue.append(nxt)

print(visited)

**(4-3) 트레이싱 - 큐 상태** 아래 코드는 매 단계 큐의 내용을 찍는다.
실행 전에 큐가 어떻게 변할지 종이에 적고 비교하라.

In [ ]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

visited = [0]
queue = [0]
while queue:
    print("큐:", queue)
    v = queue.pop(0)
    for nxt in graph[v]:
        if nxt not in visited:
            visited.append(nxt)
            queue.append(nxt)

**정답:**
```
큐: [0]
큐: [1, 2]
큐: [2, 3, 4]
큐: [3, 4, 5]
큐: [4, 5]
큐: [5]
```
큐 길이가 늘었다가 줄어든다. 늘어난 만큼이 "그 층에 있는 정점 수"다.

---
## 5. BFS로 최단 거리 구하기

BFS는 가까운 것부터 방문한다. 그래서 **처음 도달했을 때가 곧 최단 거리**다.
`visited` 리스트 대신 **거리 딕셔너리 `dist`** 를 쓰면 방문 여부와 거리를 한 번에 관리할 수 있다.

- `nxt not in dist` : `dist`에 그 정점 키가 아직 없다 = 아직 방문 안 함
- `dist[nxt] = dist[v] + 1` : 이웃은 나보다 한 칸 더 멀다

딕셔너리에 `in`을 쓰면 **키가 있는지**를 확인한다. 값이 아니다.

DFS로는 이게 안 된다. DFS는 먼 길로 먼저 들어갈 수 있어서 처음 도달한 경로가 최단이라는 보장이 없다.

**(5-1) 빈칸 - 거리 갱신** 두 곳을 채워라.

In [ ]:
# [직접 풀기]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

dist = {0: 0}      # 시작점은 거리 0
queue = [0]
while queue:
    v = queue.pop(0)
    for nxt in graph[v]:
        if ____:               # 아직 방문 안 했으면
            dist[nxt] = ____   # 나보다 한 칸 멀다
            queue.append(nxt)

# print(dist)   # {0: 0, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2}

In [ ]:
# [정답]
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

dist = {0: 0}
queue = [0]
while queue:
    v = queue.pop(0)
    for nxt in graph[v]:
        if nxt not in dist:        # 키가 없으면 아직 방문 전
            dist[nxt] = dist[v] + 1
            queue.append(nxt)

print(dist)
print(dist[5])   # 0에서 5까지 최단 거리

**(5-2) 출력 예측** 시작점을 3번으로 바꾸면 `dist`는 어떻게 될까? **키가 찍히는 순서까지** 예측하라.

In [ ]:
graph = {
    0: [1, 2],
    1: [0, 3, 4],
    2: [0, 5],
    3: [1],
    4: [1, 5],
    5: [2, 4],
}

dist = {3: 0}
queue = [3]
while queue:
    v = queue.pop(0)
    for nxt in graph[v]:
        if nxt not in dist:
            dist[nxt] = dist[v] + 1
            queue.append(nxt)

print(dist)   # 예측: ?

**정답:** `{3: 0, 1: 1, 0: 2, 4: 2, 2: 3, 5: 3}`
```
3 -> 1        : 거리 1
1 -> 0, 4     : 거리 2
0 -> 2        : 거리 3
4 -> 5        : 거리 3
```
**딕셔너리는 넣은 순서대로 출력된다.** 그래서 키가 `0,1,2,...` 로 정렬되어 나오지 않고
BFS가 방문한 순서인 `3, 1, 0, 4, 2, 5` 로 나온다.

5번까지 가는 길은 `3-1-4-5`(3칸)와 `3-1-0-2-5`(4칸)가 있는데 BFS는 짧은 쪽 3을 준다.

---
## 6. 격자 탐색 (섬 세기)

2차원 리스트도 그래프다. **각 칸이 정점, 상하좌우 인접 칸이 이웃**이라고 보면 된다.
이웃이 좌표 계산으로 바로 나오므로 여기서는 `graph` 딕셔너리를 따로 만들지 않는다.

```python
dr = [-1, 1, 0, 0]   # 위, 아래
dc = [0, 0, -1, 1]   # 왼쪽, 오른쪽
for d in range(4):
    nr = r + dr[d]
    nc = c + dc[d]
```

격자에서는 **범위를 벗어나는지 반드시 확인**해야 한다. `-1`이나 크기 이상의 인덱스로 접근하면
파이썬은 에러를 내거나(양수 초과) 엉뚱한 반대편 칸을 읽는다(음수).

**(6-1) 빈칸 - 범위 검사** 좌표 `(r, c)`가 격자 안에 있는지 판정하는 조건을 채워라.

In [ ]:
# [직접 풀기]
grid = [[1,1,0,0],
        [1,0,0,1],
        [0,0,1,1],
        [0,0,1,0]]
H = len(grid)      # 행 수 4
W = len(grid[0])   # 열 수 4

def inside(r, c):
    return ____

# print(inside(0, 0), inside(3, 3))    # True True
# print(inside(-1, 0), inside(0, 4))   # False False

In [ ]:
# [정답]
grid = [[1,1,0,0],
        [1,0,0,1],
        [0,0,1,1],
        [0,0,1,0]]
H = len(grid)
W = len(grid[0])

def inside(r, c):
    return 0 <= r < H and 0 <= c < W

print(inside(0, 0), inside(3, 3))    # True True
print(inside(-1, 0), inside(0, 4))   # False False

**(6-2) 출력 예측 - 한 덩어리 칠하기** `(0,0)`에서 DFS로 이어진 `1`을 모두 `2`로 바꾼다.
실행 후 격자가 어떻게 될지 예측하라.

```
1 1 0 0
1 0 0 1
0 0 1 1
0 0 1 0
```

In [ ]:
grid = [[1,1,0,0],
        [1,0,0,1],
        [0,0,1,1],
        [0,0,1,0]]
H, W = len(grid), len(grid[0])
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

def paint(r, c):
    grid[r][c] = 2                     # 방문 표시 겸 색칠
    for d in range(4):
        nr, nc = r + dr[d], c + dc[d]
        if 0 <= nr < H and 0 <= nc < W and grid[nr][nc] == 1:
            paint(nr, nc)

paint(0, 0)
for row in grid:
    print(row)

**정답:**
```
[2, 2, 0, 0]
[2, 0, 0, 1]
[0, 0, 1, 1]
[0, 0, 1, 0]
```
`(0,0)`과 이어진 것은 `(0,1)`, `(1,0)` 세 칸뿐이다. 오른쪽 아래 덩어리는 대각선으로만 닿아 있어
상하좌우 기준으로는 **이어져 있지 않다.**

여기서 `grid[r][c] = 2` 가 `visited` 역할을 한다. 별도 리스트 없이 격자 자체에 표시하는 흔한 요령이다.

**(6-3) 전체 작성 - 섬 개수 세기** `1`로 이루어진 덩어리(섬)가 몇 개인지 세어라.
모든 칸을 훑다가 아직 `1`인 칸을 만나면 거기서 DFS를 돌려 덩어리 전체를 지우고, 횟수를 센다.

In [ ]:
# [직접 풀기]
grid = [[1,1,0,0],
        [1,0,0,1],
        [0,0,1,1],
        [0,0,1,0]]
H, W = len(grid), len(grid[0])
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

def paint(r, c):
    grid[r][c] = 0
    for d in range(4):
        nr, nc = r + dr[d], c + dc[d]
        if 0 <= nr < H and 0 <= nc < W and grid[nr][nc] == 1:
            paint(nr, nc)

count = 0
# 여기에 이중 for문을 작성하세요

# print(count)   # 2

In [ ]:
# [정답]
grid = [[1,1,0,0],
        [1,0,0,1],
        [0,0,1,1],
        [0,0,1,0]]
H, W = len(grid), len(grid[0])
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

def paint(r, c):
    grid[r][c] = 0
    for d in range(4):
        nr, nc = r + dr[d], c + dc[d]
        if 0 <= nr < H and 0 <= nc < W and grid[nr][nc] == 1:
            paint(nr, nc)

count = 0
for r in range(H):
    for c in range(W):
        if grid[r][c] == 1:   # 아직 안 지운 덩어리를 발견
            paint(r, c)       # 덩어리 전체를 0으로 지우고
            count = count + 1 # 섬 하나 셈

print(count)   # 2

**(6-4) 응용 - BFS로 최단 이동 거리** `0`이 길, `1`이 벽인 미로에서
`(0,0)`부터 `(3,3)`까지 최소 몇 칸 움직여야 하는지 BFS로 구하라. (시작 칸 포함해서 세면 7)

거리는 **좌표 `(r, c)`를 키로 하는 딕셔너리**에 담는다. 5단원과 똑같은 코드다.

In [ ]:
# [직접 풀기]
maze = [[0,0,1,0],
        [1,0,1,0],
        [0,0,0,0],
        [0,1,1,0]]
H, W = len(maze), len(maze[0])
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

dist = {(0, 0): 1}      # 시작 칸을 1로 세기 시작

# 여기에 BFS를 작성하세요

# print(dist[(3, 3)])   # 7

In [ ]:
# [정답]
maze = [[0,0,1,0],
        [1,0,1,0],
        [0,0,0,0],
        [0,1,1,0]]
H, W = len(maze), len(maze[0])
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

dist = {(0, 0): 1}
queue = [(0, 0)]
while queue:
    r, c = queue.pop(0)
    for d in range(4):
        nr, nc = r + dr[d], c + dc[d]
        if 0 <= nr < H and 0 <= nc < W:
            if maze[nr][nc] == 0 and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                queue.append((nr, nc))

print(dist[(3, 3)])   # 7

# 격자 모양으로 보기 (못 가는 칸은 -1)
for r in range(H):
    print([dist.get((r, c), -1) for c in range(W)])

---
## 정리

|  | DFS | BFS |
|---|---|---|
| 자료구조 | 스택 (또는 재귀) | 큐 |
| 꺼내는 위치 | `pop()` 맨 뒤 | `pop(0)` 맨 앞 |
| 탐색 모양 | 한 길로 깊이 파고듦 | 가까운 곳부터 한 겹씩 |
| 최단 거리 | 보장 안 됨 | **보장됨** |
| 잘 쓰는 곳 | 연결 덩어리 찾기, 모든 경우 훑기 | 최단 거리, 최소 횟수 |

- 그래프는 **`{정점: 이웃 목록}` 딕셔너리**로 적는다. 정점 번호가 연속이 아니어도, 이름표여도 그대로 동작한다.
- `for v in graph:` 는 **키**를 돈다. `len(graph)` 는 **정점 수**다.
- 둘 다 뼈대는 같다. **꺼내는 위치만 다르다.**
- `visited`를 빼먹으면 무한 반복. DFS 재귀는 `RecursionError`, 반복 버전은 영원히 안 끝난다.
- BFS는 **큐에 넣을 때** 방문 표시한다. 꺼낼 때 표시하면 같은 정점이 큐에 여러 번 들어간다.
- 2차원 격자도 그래프다. 이웃은 `dr`/`dc` 배열로 구하고 **범위 검사를 반드시** 한다.
